# Advanced Number Theory

The **number theory** notebook covered the everyday toolkit — `math.gcd`, the sieve, `pow(a, b, m)`, Miller–Rabin. This is its sequel: three techniques that turn up in harder competitive-programming and cryptography problems, each one a clever way to *skip work* that a naive loop would grind through.

The thread tying them together is **structure**: a linear recurrence is secretly a matrix (so `n` steps collapse to `O(log n)`); an impartial game is secretly a single XOR; and a discrete logarithm is secretly a meet-in-the-middle table lookup. As always, every claim below is checked against a brute-force oracle.

**Contents**

1. **Matrix exponentiation** — a recurrence as a matrix power, O(log n)
2. **Game theory** — Nim, Sprague–Grundy, and the mex
3. **Discrete logarithm** — baby-step giant-step in O(√m)
4. When to use + cheat-sheet

## 1. Matrix exponentiation

A **linear recurrence** like Fibonacci, `F(n) = F(n-1) + F(n-2)`, is usually computed in O(n) by stepping forward. But the step is *linear*, so it can be written as a matrix acting on a state vector:

$$
\begin{bmatrix} F(n) \\ F(n-1) \end{bmatrix}
=
\begin{bmatrix} 1 & 1 \\ 1 & 0 \end{bmatrix}
\begin{bmatrix} F(n-1) \\ F(n-2) \end{bmatrix}
\quad\Longrightarrow\quad
\begin{bmatrix} F(n+1) \\ F(n) \end{bmatrix}
=
\begin{bmatrix} 1 & 1 \\ 1 & 0 \end{bmatrix}^{\,n}
\begin{bmatrix} 1 \\ 0 \end{bmatrix}.
$$

Applying the matrix `n` times is just raising it to the `n`-th power — and a power can be computed by **binary (squaring) exponentiation**, exactly the trick `pow(b, e, m)` uses for integers (the **number theory** notebook). That turns O(n) into **O(log n)** matrix multiplies. With a fixed matrix size `k`, each multiply is O(k³), so the whole thing is O(k³ log n).

First the two primitives: matrix multiply, and matrix power by squaring. An optional `mod` keeps entries small when the answer is wanted modulo some prime — the same use case as integer `pow(b, e, m)`.

In [1]:
def mat_mul(A, B, mod=None):
    """Multiply two matrices, optionally reducing entries mod `mod`."""
    n, kk, p = len(A), len(B), len(B[0])
    C = [[0] * p for _ in range(n)]
    for i in range(n):
        Ai, Ci = A[i], C[i]
        for k in range(kk):
            a = Ai[k]
            if a:                                    # skip zero rows of the recurrence
                Bk = B[k]
                for j in range(p):
                    Ci[j] += a * Bk[j]
                    if mod is not None:
                        Ci[j] %= mod
    return C

def mat_pow(M, e, mod=None):
    """M ** e by binary exponentiation: O(log e) multiplies."""
    n = len(M)
    R = [[int(i == j) for j in range(n)] for i in range(n)]   # identity
    while e:
        if e & 1:                                    # fold current square into result
            R = mat_mul(R, M, mod)
        M = mat_mul(M, M, mod)                        # square the base
        e >>= 1
    return R

print("[[1,1],[1,0]] ** 5 =", mat_pow([[1, 1], [1, 0]], 5))   # top-left entry is F(6)=8


[[1,1],[1,0]] ** 5 = [[8, 5], [5, 3]]


### Fast Fibonacci

With those, Fibonacci is a one-liner: raise `[[1,1],[1,0]]` to the `n` and read off an entry. We prove it agrees with a `functools.lru_cache` memoized version for every `n` up to 30, and matches on a value far too large to want to compute step-by-step.

In [2]:
from functools import lru_cache

def fib_matrix(n):
    if n == 0:
        return 0
    return mat_pow([[1, 1], [1, 0]], n)[0][1]         # = M[1][0] = F(n)

@lru_cache(maxsize=None)
def fib_memo(n):
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

assert all(fib_matrix(n) == fib_memo(n) for n in range(31))
print("fib_matrix == fib_memo for n in 0..30: OK")
print("F(100) =", fib_matrix(100))
assert fib_matrix(100) == fib_memo(100)              # 21-digit value, O(log 100) work
print("matches memoized F(100):", fib_matrix(100) == fib_memo(100))


fib_matrix == fib_memo for n in 0..30: OK
F(100) = 354224848179261915075
matches memoized F(100): True


### Any k-term recurrence

The same idea generalizes. For `x(n) = c₁·x(n-1) + c₂·x(n-2) + … + c_k·x(n-k)`, the **companion matrix** has the coefficients across its top row and an identity shifted one step below the diagonal — multiplying by it shifts the state window forward by one. Raise it to a power and you can jump straight to `x(n)`.

We test the framework two ways: against the naive O(n) **tribonacci** (`T(n)=T(n-1)+T(n-2)+T(n-3)`) and against a custom recurrence `a(n) = 2·a(n-1) + 3·a(n-2)`.

In [3]:
def linrec(coeffs, init, n):
    """x(n) for x(n) = sum(coeffs[i] * x(n-1-i)); init = [x0, x1, ..., x_{k-1}]."""
    k = len(coeffs)
    if n < k:
        return init[n]
    M = [[0] * k for _ in range(k)]
    M[0] = list(coeffs)                               # top row: the coefficients
    for i in range(1, k):
        M[i][i - 1] = 1                               # sub-diagonal identity: shift window
    Mp = mat_pow(M, n - (k - 1))
    state = [init[k - 1 - i] for i in range(k)]       # [x_{k-1}, ..., x_0]
    return sum(Mp[0][j] * state[j] for j in range(k))

def trib_naive(n):                                    # O(n) reference
    seq = [0, 1, 1]
    while len(seq) <= n:
        seq.append(seq[-1] + seq[-2] + seq[-3])
    return seq[n]

def custom_naive(n):                                  # a(n) = 2a(n-1) + 3a(n-2)
    a, b = 1, 2
    for _ in range(n):
        a, b = b, 2 * b + 3 * a
    return a

assert all(linrec((1, 1, 1), (0, 1, 1), n) == trib_naive(n) for n in range(25))
assert all(linrec((2, 3), (1, 2), n) == custom_naive(n) for n in range(25))
print("tribonacci  linrec == naive for n in 0..24: OK")
print("custom 2,3  linrec == naive for n in 0..24: OK")
print("T(50) =", linrec((1, 1, 1), (0, 1, 1), 50))
print("a(30) =", linrec((2, 3), (1, 2), 30))


tribonacci  linrec == naive for n in 0..24: OK
custom 2,3  linrec == naive for n in 0..24: OK
T(50) = 5742568741225
a(30) = 154418349070987


The payoff is purely asymptotic: for small `n` the naive loop wins (no matrix overhead), but the matrix version is O(log n), so for astronomically large `n` — or when you need `x(n) mod p` for `n` up to 10¹⁸ — it is the only option. Pass `mod` through `mat_pow` and every entry stays bounded.

## 2. Game theory — impartial games

An **impartial game** is one where, from any position, both players have exactly the same set of moves (so it doesn't matter *who* is to move, only the position). Under **normal play** — the player who *cannot* move loses — every such game has a startlingly clean theory.

The base case is **Nim**: several piles of stones, a move removes one or more stones from a single pile. The remarkable result (Bouton, 1901): the player to move wins **iff the XOR of all pile sizes is nonzero**. That XOR is called the *nim-sum* — the same bitwise XOR from the **bit manipulation** notebook, here carrying deep game-theoretic meaning.

We verify Bouton's theorem the only honest way: against a brute-force **minimax** that recursively asks "does some move lead to a position the opponent loses?" — checked over *every* position with three piles of size ≤ 4.

In [4]:
from functools import lru_cache, reduce
from operator import xor

def nim_first_player_wins(piles):
    return reduce(xor, piles, 0) != 0                 # nim-sum nonzero => win

@lru_cache(maxsize=None)
def nim_brute(piles):
    """Minimax: current player wins iff some move reaches a losing position."""
    for i, p in enumerate(piles):
        for take in range(1, p + 1):                  # remove 1..p from pile i
            nxt = list(piles)
            nxt[i] -= take
            nxt = tuple(sorted(x for x in nxt if x))  # canonical: drop empties, sort
            if not nim_brute(nxt):
                return True
    return False                                      # no move => current player loses

checks = 0
for a in range(5):
    for b in range(5):
        for c in range(5):
            piles = tuple(sorted(x for x in (a, b, c) if x))
            assert nim_first_player_wins(piles) == nim_brute(piles)
            checks += 1
print(f"Nim XOR verdict == brute minimax for all {checks} positions (3 piles <=4): OK")
print("XOR(3,4,5) =", 3 ^ 4 ^ 5, "-> first player",
      "WINS" if nim_first_player_wins((3, 4, 5)) else "LOSES")
print("XOR(1,2,3) =", 1 ^ 2 ^ 3, "-> first player",
      "WINS" if nim_first_player_wins((1, 2, 3)) else "LOSES")


Nim XOR verdict == brute minimax for all 125 positions (3 piles <=4): OK
XOR(3,4,5) = 2 -> first player WINS
XOR(1,2,3) = 0 -> first player LOSES


### Sprague–Grundy and the mex

Nim is not a special case — it is *the* case. The **Sprague–Grundy theorem** says every impartial game position has a **Grundy number** (nim-value), and the position behaves exactly like a single Nim pile of that size. Two consequences:

- A position is **losing** for the mover iff its Grundy number is **0**.
- The Grundy number of a *sum* of independent games is the **XOR** of their Grundy numbers — so multi-component games reduce to Nim on the components' values.

The Grundy number is defined by the **mex** (minimum excludant): the smallest non-negative integer *not* among the Grundy numbers of the positions you can move to. `mex({0,1,3}) = 2`. We compute Grundy values for a **subtraction game** (remove 1, 3, or 4 stones) and check the rule against brute minimax — both for a single pile and for a two-pile sum, where the XOR rule does the heavy lifting.

In [5]:
MOVES = (1, 3, 4)                                      # stones you may remove

@lru_cache(maxsize=None)
def grundy(n):
    reachable = {grundy(n - s) for s in MOVES if n - s >= 0}
    m = 0
    while m in reachable:                              # mex: smallest int not reachable
        m += 1
    return m

@lru_cache(maxsize=None)
def sub_brute(n):                                      # single-pile minimax
    return any(n - s >= 0 and not sub_brute(n - s) for s in MOVES)

# single pile: Grundy 0 <=> losing position
assert all((grundy(n) != 0) == sub_brute(n) for n in range(60))
print("single pile: (grundy != 0) == brute win, n in 0..59: OK")
print("grundy(0..11):", [grundy(n) for n in range(12)])

@lru_cache(maxsize=None)
def multi_brute(piles):                               # two-pile sum, minimax oracle
    for i, p in enumerate(piles):
        for s in MOVES:
            if p - s >= 0:
                nxt = list(piles); nxt[i] -= s
                nxt = tuple(sorted(x for x in nxt if x))
                if not multi_brute(nxt):
                    return True
    return False

checks = 0
for a in range(9):
    for b in range(9):
        piles = tuple(sorted(x for x in (a, b) if x))
        nim_sum = reduce(xor, (grundy(p) for p in piles), 0)
        assert (nim_sum != 0) == multi_brute(piles)
        checks += 1
print(f"two-pile sum: (XOR of grundy != 0) == brute win for {checks} positions: OK")


single pile: (grundy != 0) == brute win, n in 0..59: OK
grundy(0..11): [0, 1, 0, 1, 2, 3, 2, 0, 1, 0, 1, 2]
two-pile sum: (XOR of grundy != 0) == brute win for 81 positions: OK


That second assert is the whole point of Sprague–Grundy: we never built a minimax over the *combined* game's huge state space — we computed each pile's Grundy number independently and **XOR**ed them, and it matched the brute oracle exactly. That is how you solve game-sum problems that are otherwise exponential.

## 3. Discrete logarithm — baby-step giant-step

The **discrete logarithm problem**: given `a`, `b`, `m`, find `x` with `a^x ≡ b (mod m)`. It is the inverse of modular exponentiation — and its presumed hardness for large `m` underpins Diffie–Hellman and ElGamal. A brute-force scan of `x = 0, 1, 2, …` is O(m).

**Baby-step giant-step** (Shanks) is a meet-in-the-middle that cuts that to **O(√m)** time and space. Write `x = i·n + j` with `n = ⌈√m⌉` and `0 ≤ j < n`. Then `a^x = b` becomes `a^j = b·(a^{-n})^i`. Precompute all *baby steps* `a^j` in a table, then take *giant steps* `b·(a^{-n})^i` until one lands in the table — meeting in the middle (the **meet-in-the-middle** pattern). Both phases are O(√m).

The implementation needs the modular inverse `a^{-n} mod m` — `pow(a, -n, m)`, the 3.8+ builtin from the **number theory** notebook (valid when `gcd(a, m) = 1`). We store the *smallest* `j` for each baby step so the recovered `x` is the smallest non-negative solution.

In [6]:
import math

def bsgs(a, b, m):
    """Smallest x >= 0 with a**x == b (mod m), or None. Requires gcd(a, m) == 1."""
    a, b = a % m, b % m
    n = math.isqrt(m) + 1                             # block size ceil(sqrt(m))
    table = {}                                        # baby steps a**j -> smallest j
    cur = 1
    for j in range(n):
        table.setdefault(cur, j)
        cur = cur * a % m
    factor = pow(a, -n, m)                            # a**(-n): the giant stride
    gamma = b
    for i in range(n + 1):                            # giant steps b * factor**i
        if gamma in table:
            return i * n + table[gamma]
        gamma = gamma * factor % m
    return None

def brute_dlog(a, b, m):                              # O(m) reference: smallest x
    a, b, cur = a % m, b % m, 1 % m
    for x in range(m):
        if cur == b:
            return x
        cur = cur * a % m
    return None

# exhaustive check vs brute force over several prime moduli
checks = 0
for m in (7, 11, 13, 101, 197, 1009):
    for a in range(2, min(m, 25)):
        if math.gcd(a, m) != 1:
            continue
        for x in range(min(m, 20)):
            b = pow(a, x, m)
            assert bsgs(a, b, m) == brute_dlog(a, b, m)
            checks += 1
print(f"bsgs == brute-force smallest x for {checks} cases: OK")


bsgs == brute-force smallest x for 1657 cases: OK


And the speedup is the real prize — a discrete log modulo a near-million prime, found in a √m-sized table lookup instead of a million-step scan:

In [7]:
m, a, x_true = 1_000_003, 5, 987_654         # m prime; hide x_true inside b
b = pow(a, x_true, m)
x = bsgs(a, b, m)
print(f"solved  5^x == {b}  (mod {m})")
print(f"recovered x = {x}")
print("verify  pow(5, x, m) == b:", pow(a, x, m) == b)
assert pow(a, x, m) == b
print(f"table size ~ sqrt(m) = {math.isqrt(m) + 1}  vs brute scan of {m}")


solved  5^x == 265179  (mod 1000003)
recovered x = 987654
verify  pow(5, x, m) == b: True
table size ~ sqrt(m) = 1001  vs brute scan of 1000003


## 4. When to use what

| You need… | Reach for | Why |
|---|---|---|
| The `n`-th term of a linear recurrence, huge `n` | matrix exponentiation | O(log n) instead of O(n); works mod p |
| Count paths / tilings via a fixed transfer rule | matrix exponentiation | the transfer matrix raised to a power |
| Who wins a take-away / Nim-like game | nim-sum (XOR) | Bouton's theorem, O(#piles) |
| Who wins a general impartial game | Sprague–Grundy + mex | reduce each component to a nim-value, XOR them |
| Solve `a^x ≡ b (mod m)`, moderate `m` | baby-step giant-step | O(√m) meet-in-the-middle vs O(m) scan |
| Solve a discrete log for crypto-sized `m` | (not these) | BSGS's O(√m) is still infeasible — that's the security |

**Cross-references:** the integer `pow(b, e, m)`, modular inverse `pow(a, -1, m)`, and primality live in the **number theory** notebook; the XOR nim-sum is the bitwise XOR of the **bit manipulation** notebook; the split-and-meet structure of BSGS is the **meet-in-the-middle** pattern.

## 5. Cheat-sheet

| Technique | Problem | Complexity | Key idea |
|---|---|:---:|---|
| Matrix exponentiation | k-term linear recurrence, n-th term | O(k³ log n) | companion matrix `**` n by squaring |
| Nim | take-from-one-pile game | O(#piles) | win ⟺ XOR of piles ≠ 0 |
| Sprague–Grundy | any impartial game | O(states · moves) | Grundy = mex of moves; sum = XOR |
| mex | building block of Grundy | O(#reachable) | smallest absent non-negative int |
| Baby-step giant-step | `a^x ≡ b (mod m)` | O(√m) time & space | `x = i·n + j`, meet in the middle |

**The unifying trick** is replacing a linear scan with structure: a recurrence becomes a matrix power (log instead of linear), an impartial game collapses to one XOR, and an O(m) search becomes two O(√m) passes that meet in the middle.